# House Price Prediction and Model Comparison
- Collaborators: Valeria Garcia Hernandez, An Nguyen
- CSE 2107 Final Project

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

## Introduction
We will take a look at data for Melbourne Housing Price Prediction https://www.kaggle.com/datasets/dansbecker/melbourne-housing-snapshot. This project aims at building a model of housing prices to predict median house values in Melbourne using the provided dataset. This model should learn from the data and be able to predict the median housing price in any district, given all the other metrics.

Our goal will be to use this dataset to gain some insight about characteristics of different features. We will be using data profiling from pandas and create models and compare them to find out which model performed the best and probably explain why it is the best model among all in this data.

In [ ]:
# download the data and load it in this notebook for further processing and analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import linear_model
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import style
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
data = 'utility/data'     

assert os.path.exists(data), 'Unable to find data'

melb_housing = pd.read_csv(f'{data}/melb_data.csv', low_memory=False)

melb_housing.info()

# 1. Describing the Data
The Melbourne Housing Snapshot is a Kaggle dataset who's creator is Tony Pino. The dataset was obtained by scraping from publicly available results of Domain.com.au. 

Examples & Features: The dataset contains 13580 rows and 21 columns, so 13580 observations and 21 features. 

## Feature Summary
- **Suburb** (categorical, nominal) - Name of suburb where the property is located.
- **Address** (categorical, nominal) - Street address of the property. 
- **Rooms** (numerical, discrete) - Number of primary rooms.
- **Type** (categorical, nominal) - Property type.
- **Price** (numerical, continuous) - Sold price of the property.
- **Method** (categorical, nominal) - Method of sale. 
- **SellerG** (categorical, nominal) - Real estate seller or group. 
- **Date** (categorical -- time, interval) - Date the property was sold.
- **Distance** (numerical, continuous) - Distance from the Central Business District (CBD) in kilometers. 
- **Postcode** (numerical, discrete) - The property's postcode.
- **Bedroom2** (numerical, discrete) - The number of bedrooms. 
- **Bathroom** (numerical, discrete) - The number of bathrooms. 
- **Car** (numerical, discrete) - The number of car parking spaces.
- **LandSize** (numerical, continuous) - The size of the land in square meters. 
- **BuildingArea** (numerical, continuous) - The size of the building area in square meters. 
- **YearBuilt** (numerical, discrete) - The year the property was built. 
- **CouncilArea** (categorical, nominal) - The local government area/council the property belongs to.
- **Lattitude** (numerical, continuous) - The geographical latitude coordinate. 
- **Longtitude** (numerical, continuous) - The geographical longitude coordinate. 
- **Regionname** (categorical, nominal) - The general region of the city where the property is located. 
- **Propertycount** (numerical, discrete) - The total number of properties in that suburb. 

## Overview
**Type of Problem:** Supervised Learing.
Supervised learning requires a labeled dataset which means each data point (house) has an associated target value (the Price). This is a supervised learning problem focused on predictive modeling of real estate prices. 

**Application Area:** Real Estate/Housing Market Analysis. 
The dataset contains property characteristics (Suburb, Rooms, Landszide), geographical information (Distance, Lattitude, Longtitude), and sale details (Price, Date, Method), which can be used to understand and analyze real estate and housing valuation. 

**Type of Prediction:** Regression. 
Our goal is to build a model that predicts median house values (Price). Regression models are used when the target variable that is to be predicted is a continuous numeric value. The target variable, Price, is a continuous float64 (numerical, continuous variable). The model aims to predict a specific price value, rather than a category or group.

### Getting Familiar with the Data

In [ ]:
# Note: ydata_profiling may not run in newer Python versions 
# This section was included as part of exploratory data analysis
#!pip install numba==0.58.1
#!pip install ydata_profiling==4.1.2
#!pip install ipywidgets jupyterlab_widgets

In [ ]:
from ydata_profiling import ProfileReport

profile = melb_housing.profile_report(
    correlations={
        "pearson": {"calculate": True},
        "spearman": {"calculate": False},
        "kendall": {"calculate": False},
        "phi_k": {"calculate": False},
        "cramers": {"calculate": False}
    }
)
profile.to_notebook_iframe()

# Data Quality and Structure Assessments
The dataset offers a snapshot of the Melbourne housing market, comprising of detailed sale records for 13,580 properties. This section is intended to offer detailed descriptions of any major highlights/insights found in the dataset, aiming to provide a better understanding of the data.

## I. Feature Distributions
We are looking at the feature distributions in correlation to our target variable, Price.

#### Target Variable: Price

In [ ]:
%matplotlib inline

sns.set_style('whitegrid')
sns.set_palette("colorblind") 

fig, axes = plt.subplots(2, 1, figsize=(10, 10))

# Histogram: to show skew and full distribution. 
sns.histplot(melb_housing["Price"], bins=60, ax=axes[0])
axes[0].set_title("Price Distribution (Histogram)")
axes[0].set_xlabel("Price")
axes[0].set_ylabel("Frequency")

# Boxplot: to show outliers. 
sns.boxplot(x=melb_housing["Price"], ax=axes[1])
axes[1].set_title("Price Distribution (Boxplot)")
axes[1].set_xlabel("Price")

plt.tight_layout()
plt.show()


**Price Distribution (Histogram):** The histogram of property prices shows a strong right skew which indicates that while most properties are priced in the lower-to-mid range, a smaller number of very expensive houses extend the distribution to the right. This can suggest that the Melbourne housing market includes a large volume of moderately priced homes and a limited number of luxury, high-value properties. The distribution doesn't appear normal, and the presence of extreme values seems to affect the overall shape.

**Price Distribution (Box Plot):** The boxplot highlights a large number of outliers which confirms the skew seen in the histogram. The median price is relatively low compared to the upper extreme values, and the interquartile range (IQR) looks narrow compared to the full price range. This could be an indicator that most properties fall within a fairly concentrated price band, and a significant number of properties lie far above the typical price range.

**Combined Interpretation:** Most Melbourne properties fall into a lower or mid-price range; a small number of very expensive houses create a long tail on the right side of the price distribution. The accompanying boxplot confirms this pattern, revealing a large number of high-value outliers. These plots together suggest that there may be benefit reducing and in price modeling from transformations that would be able to reduce skewness and better handle extreme values.

## II. Missing Data Analysis
Under Missing Values, we can see the summary statistics of which features have the most missing values in their data and **YearBuilt** as well as **BuildingArea** bubble up to be the two most prominent variables containing missingness.

In [ ]:
cols = ["YearBuilt", "BuildingArea"]

# calculating how many missing and non-missing points in columns. 
missing_counts = melb_housing[cols].isnull().sum()
not_missing_counts = melb_housing[cols].notnull().sum()

fig, ax = plt.subplots(figsize=(8, 5))

# creating bars for "Missing" and "Not Missing"
ax.bar(cols, missing_counts, label="Missing")
ax.bar(cols, not_missing_counts, bottom=missing_counts, label="Not Missing")

# labeling
ax.set_title("Missing Data for YearBuilt and BuildingArea")
ax.set_ylabel("Count")
ax.legend()

plt.show()

# calculating percent of missing data for features
missing_percent = (melb_housing[["YearBuilt", "BuildingArea"]].isnull().mean() * 100).round(2)
print(missing_percent)


The height of each bar indicates the proportion of missing values in that feature. Taller bars correspond to features with more missing entries which can affect modeling and require things like preprocessing to mitigate.

Both YearBuilt and BuildingArea contain substantial missing data, however, BuildingArea has a larger percentage of missing data (47.5%) than YearBuilt (39.58%) by around a 7.92% difference.

The missingness suggests that these variables may not have been consistently recorded in the original data collection process, especially BuildingArea. The magnitude of missing data can imply that things like simple row removal would lead to considerable data loss; imputation strategies and/or models capable of handling missingness may be needed.

The bar plot highlights that YearBuilt and especially BuildingArea require careful preprocessing decisions, and their high missingness may limit how much they can contribute to predictive modeling.

## III. Outliers
The Profile Report shows the characteristics of each feature, and the top four features that showcase significant outliers can be pinpointed as Price, Rooms, Landsize, and BuildingArea.

In [ ]:
%matplotlib inline

#log transform high skewed features that make visualization harder to see.
melb_housing['Log_Landsize'] = np.log1p(melb_housing['Landsize'])
melb_housing['Log_BuildingArea'] = np.log1p(melb_housing['BuildingArea'])

# Features to visualize for outliers.
features = ["Price", "Rooms", "Log_Landsize", "Log_BuildingArea"]

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(len(features), 1, figsize=(12, 14))

for ax, col in zip(axes, features):
    sns.boxplot(x=melb_housing[col], ax=ax, palette = 'Purples')
    ax.set_title(f"Boxplot of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel('Distribution')

plt.tight_layout()
plt.show()

#### Price
**Observations**: The distribution is heavily right-skewed. The box (IQR) is compressed toward the lower end of the range, while the upper whisker extends considerably further. Numerous data points lie beyond the upper whisker, representing properties priced well above the typical range.

**Interpretation**: The housing market dataset exhibits significant price inequality, with most properties concentrated in the lower-to-middle price range while a substantial number of luxury properties create extreme outliers. The high-price outliers have the ability to inflate the mean and disproportionately influence linear models. The outliers may also represent actual luxury properties like prestigous suburbs instead of data errors so careful investigation is needed before considering removal. 

#### Rooms
**Observations**: The box plot shows a strong central tendency that hovers around 2-3 rooms. Some properties extend beyond the upper whisker, with room counts of 5 or more. 

**Interpretation**: The vast majority of properties fall within the standard room count (2-3 rooms) of the box (IQR). Houses with unusually high room counts are outliers. These may correspond to mansions or very large properties. The presence of a small number of right-side outliers (rooms that exceed the maximum of the box plot) indicates a few properties that are exceptionally large compared to the standard property. These outliers are sparse and do not heavily influence the central distribution as much as the outliers of the other box plots of different features.

#### Landsize
**Observations**: After log transformation, the box seems more relativively centered, centering between approximately 5.5-6.5. Significant outliers persist on both ends: upper outliers cluster around 8-10 on the log scale, and lowers outliers are much more scatter but there is some concentration around 3. 

**Interpretation**: Even after log-transformation to reduce skewness, Landsize exhibits a large number of outliers on both ends of the spectrum. The upper outliers may represent properties with exceptionally large land areas, while the lower outliers may represent to properties with unusually small or minimal land areas. The continued presence of numerous outliers, especially the outliers beyond the upper whisker suggests that Landsize is a highly variable feature, and these extremes should be carefully considered or capped before being used in modeling.

#### BuildingArea
**Observations**: The box (IQR) looks tight, hovering around the unit of 5 on the log scale, which can indicate that 50% of the data is very concentrated. There is a large number of outliers plotted above the upper whisker, and an even greater frequency of outliers plotted below the lower whisker.

**Interpretation**: The strong concentration of the main data (the IQR) suggests that most properties have a consistent, moderate building area. The presence of numerous outliers on both sides of the plot indicates significant variability at extremes. The high frequency of points below the lower whisker suggests a large number of properties with unusually small building area, and the points above the upper whisker represent properties with exceptionally large building areas. The high number of outliers, particularly on the low end, suggests that the typical range of building area (seen by the whiskers) doesn't capture a significant portion of the dataset. For modeling, these extreme observations should be thoroughly investigated to prevent any over-influence over the model training.

## IV. Correlation and Feature Relationships
In this section, we will go over many featuers that have high correlation relationship with each other, and more specifically, features that exhibit a high correlation relationship with our target variable, Price. 

In [ ]:
# Create a correlation matrix heatmap
correlation_matrix = melb_housing.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
plt.title("Feature Relationships Heatmap")

sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin = -1, vmax = 1, center = 0, square=True,linewidths = .12, cbar_kws={"shrink": .75});
plt.show()

This is a more detailed visualization of a heatmap than the one in the Profile Report; the purpose is to help more clearly identify any correlations between features. Specific numbers and a different color palette are included for each square for added clarity. We can start pulling out features that show the strongest correlations to our target variable, Price, and a strong relationship with other features in the heatmap. 

#### Highest Positive Correlation to Target Variable: Price
**Observations:** Rooms (0.5), Bedroom2 (0.48), and Bathroom (0.47) are the top features that exhibit the biggest postive correlations with our target variable, Price; each respective feature shows a moderate positive linear correlation (with Bedroom2 and Bathroom being slightly weaker compared to Rooms). 

**Interpretation**: Features that increase a home's size, capacity, and utility generally drive up the market price. There is a clear and consistent trend for Price to rise with these features, meaning the relationship is reliable; however, since these features are not closer to 1, the relationship isn't perfect. There are other factors that go into determining Price. The coefficients of the correlations are very close together, suggesting these features are highly intercorrelated: properties with more rooms tend to have more bedrooms and bathrooms. The features seem to capture overlapping information rather than independent efforts. 

#### Highest Negative Correlation to Target Variable: Price
**Observations**: YearBuilt shows a negative correlation of -0.32 with Price, making it the weakest correlation among the major numerical features. The negative correlation indicates an inverse relationship: as the year a property was built increases (newer properties), the price tends to decrease. The negative correlation seems to be weak in strength. Additionally, YearBuilt, has significant missing data that we had seen before (39.58%), so the correlation is only calculated on an incomplete part of the dataset. 

**Interpretation**: The negatative correlation between YearBuilt and Price looks counterintutive as we'd expect newer properties to command higher pricer (more modern amenitities, better conditions, etc). However, this unexpected relationshp could be explained by several factors. This could be a reflection of the unique characteristics of the Melbourne housing market: the most desirable and most expensive properties could be located in more established, prestigious suburbs where homes were built decades ago. The older properties could be sitting in locations that are rare and valuable. There could also be the factor that newer homes may be built in surburbs where land is cheaper and properties are more affordable, dragging down the average price of recent constructions. Regardless, the weak strength (-0.32) suggests that YearBuilt by itself can be a poor predictor of price. The relationship between property value and age seems highly contextual and likely depends on other features; the substantial missing data is also concerning for interpretation. Simple correlation doesn't seem to be able to capture the entire breadth of YearBuilt.

#### High Correlation Between Features (not the Target Variable)
Let's look at a few features that are not the target variable, but show a high correlated relationship with each other. 

In [ ]:
%matplotlib inline
plt.figure(figsize=(8,6))

sns.regplot(x = 'Rooms', y = 'Bedroom2', data = melb_housing, color = 'green');
plt.title("Rooms vs. Bedrooms Regression Scatter Plot")
plt.xlabel('Number of Rooms')
plt.ylabel('Number of Bedrooms')
plt.show()

#### Rooms vs. Bedrooms Relationship
**Observations**: From looking at the heatmap, Bedroom2 and Rooms have the highest correlation with each other with a positive correlation of 0.94, which is extremely close to 1. We can see this relationship clearly represented by the regression scatter plot that shows a strong, positive linear relationship between the two features. The scatter plot represents low bias, but high variance. The regression line captures the general positive trend in the data reasonably well (as Rooms increases, Bedrooms increases). However, there is a presence of high variance because there's a noticable scatter around the regression line throughout the entire range. Points deviate from the fitted line both above and below, yet the spread still appears roughly consistent. There are also notable outliers that exist (for example, the data point at 3 rooms and 20 bedrooms).

**Interpretation**: The high variance does not come off necessarily problematic as Rooms and Bedrooms (Bedroom2) are related but not identical concepts. Rooms includes all rooms (living, dining, kitchen, etc), while Bedrooms is a subset. Any extreme outliers should be investigated as potential data quality issues. 

The most important takeaway from this data analysis should be that both Rooms and Bedroom2 are highly correlated with each other. The effects of two variables being highly correlated with one another is that when we are modeling, the model can struggle to isolate the unique effect of each highly correlated variable on the target variable (obscuring interpretation). 

When two features are extremely correlated with one another (a correlation > 0.9), only one of the features need to be kept. Dropping redundant features allows us to simplify our model without losing any type of meaningful predictive power (since both variables convey essentially the same information). This can help improve things like model interpretability, stability, and generalizations. 

## V. Categorical Insights
To provide a complete and holistic understanding of our data's major highlights and insights, it's necessary to include a dedicated section discussing any noticable analyses that numerical correlation alone cannot capture (specifically, for our target variable: Price). 

In [ ]:
# Calculate the count of each unique value in the 'Type' column/feature
# normalize = True to give percentage (instead of raw count)

# 1. Property Type
print("Property Type")
print(melb_housing['Type'].value_counts(normalize=False))
print("\n")

# 2. Selling Method
print("Selling Method")
print(melb_housing['Method'].value_counts(normalize=False))
print("\n")

# 3. Region Names
print("Region Names")
print(melb_housing['Regionname'].value_counts(normalize=False))

**Abbreviations Glossary**:

_Property Type_
- h = Houses
- u = Units/Apartments
- t = Townhouses

_Selling Method_ 

- S: property sold

- SP: property sold prior

- PI: property passed in (did not sell at auction)

- VB: vendor bid

- SA: sold after auction

#### Property Type
**Observations & Interpretation**: The dataset is overwhelmingly dominated by Houses ('h') with a count of 9022. Units ('u') comes in second most common at 3017, and Townhouses are the least common with a count of 1114. 

**Insight**: Any conclusions drawn from the market are going to be heavily skewed towards Houses because that where most of the data comes from. A higher distribution of the dataset is allocated to Houses compared to other types of properties like Units and Townhouses.

#### Selling Method
**Observations & Interpretation**: The standard Sold ('S', 9022) method is by far the most common way properties are sold. A combined total of 3,267 sales were either Sold Prior ('SP', 1,703) or Passed In ('PI', 1,564), indicating that a noticable amount of the market requires negotiation outside of the auction event to achieve a final sale. The number of Vendor Bids ('VB', 1199) shows that there are sellers actively driving up prices at auction. 

**Insight**: While the standard sale is most common, the data suggests a competitive and strategic market where auction-related activity (SP, PI, VB, SA) is prevalent, and negotiation outside of the main auction event is a discernible occurrence.

#### Region Names
**Observations & Interpretation**: The vast majority of the sales volume is concentrated in metropolitan areas as Southern Metropolitan (4695), Northern Metropolitan (3890), Western Metropolitan (2948), Eastern Metropolitan (1471), and South-Eastern Metropolitan (450) all dominate the top five of the value counts. The combined total for all the "Victoria" regions is only around 126 sales, making their presence not as prevalent in the dataset -- their sampling is tiny compared to the metroplitan regions. 

**Insight**: The dataset is mostly in the inner-city/suburban market, so any general conclusions and/or price modeling done will be reflected by the characteristics of major urban areas because there is significant sampling bias that offers a lot of metropolitan data, but less so for the less-urban regions.

### Split the Data
Extract input (X) and output (y) data from the dataset. 

Use the train and test split from the scikit learn library to split the data into 80% training and 20% testing.

Then, initalize the Kfolds. Be sure to use these folds when executing cross validation below so that the results are comparable.

In [ ]:
# Seperate featuers X, and our target y; X containing all features except 'Price.'
X = melb_housing.drop('Price', axis=1)

# y only containing target variable. 
y = melb_housing['Price']

# Splitting data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 1)

# Initialize K-Fold
kfolds = KFold(n_splits = 5, shuffle = True, random_state = 1) 



**Why It's Good Practice**: The 80% training set provides the model a large enough portion of the data to learn the underlying patterns, trends, and relationships effectively. A well-trained model requires a substantial amount of data. 

The 20% test set provides a large enough, unbiased sample that is reliable to test the model's true performance on unseen data and its generalization ability. Using too small a test set can cause the evaluation to be less trustworthy by providing something like high variance. 

**Picking Train-Test Split Size**:
- _For a small dataset:_ There may be a need to allocate a larger percentage to training to ensure the model learns effectively. However, if more of the data is allocated towards the training set, that leaves an even smaller percentage for the testing set which can make the evaluation less reliable and less representative.
  
- _For a large dataset:_ Once we're dealing with a large dataset, we can start encountering things like diminishing returns. So, using a smaller test set for testing can be sufficient in providing reliable performance estimates. Priority can be shifted to giving as much training data as possible to the split whilst also ensuring the test set is large enough to be robust.

**Purpose of Cross Validation**: The purpose of the cross validation technique is to get a more reliable, robust estimate of a model's performance than a single train-test split. (It's especially useful with smaller datasets where a single split might not be as representative of the data). 

The data is split into multiple folds (subsets). The model gets trained by a combination of these folds and validated on the remaining fold. The process repeats until each fold has been able to be used by the validation set once. The final performance measure is the average of all the validation results. 

When we train and test different subsets of the data, the technique allows for the model's performance to create a better idea on how well the model will be able to generalize when it comes to new, unseen data which ensures robustness (and prevents overfitting). 

**Data Snooping**: Data snooping (or, data leakage) is the act of when the data from the test set (or in fact, any data that needs to be unseen) leaks into the training process. This can lead to things like over-optimism or misleading estimates of the model's performance. (Essentially, the model has "cheated" by seeing some of the answers already.) 

_How to avoid:_ To avoid, we must be strict about seperating the test set. We can split first, then process second. This means that the test set does not get touched until the very final evaluation. Things like hyperparameter tuning, model selection, feature selection, etc (any types of model development where the step is learning from the data) should be done only on the training set (and the validation set, if it is being used). Once the final model is selected, its performance can be evaluated using the test set for an unbiased, final metric. 

## 2. Preprocessing and Baseline Models

### Preprocessing for the Baseline Models

Let's start by preprocessing the data so that we can train some baseline models.

# Cleaning Data
We clean the Malbourne Housing Dataset by removing unusable features and handling missing values, which creates bias and inefficiencies.

### 1. Removing Features
We remove features that are not directly relevant or need feature extraction.

In [ ]:
old_melb_housing = melb_housing.copy()
cols_to_remove = ["Address", "SellerG", "Date"]
melb_housing = melb_housing.drop(columns=cols_to_remove)
X_train = X_train.drop(columns=cols_to_remove)
X_test  = X_test.drop(columns=cols_to_remove)

**Removals:** 
Address and SellerG (Real State Agent) are not very useful for prediction.

Date needs feature extraction. In its original format as raw, Date is not particularly useful so we need to perform feature extraction on it. Extracting from Date like month and year can help extract things like general depreciation/appreciation in property over the time period of the dataset, and/or give us seasonal patterns.

### 2. Handling Missing Data

In [ ]:
# Numerical Features
melb_housing["YearBuilt"].shape[0]

#check which numerical columns are missing values and how many
numerical_cols = melb_housing.select_dtypes(include=['int64', 'float64']).columns
melb_housing[numerical_cols].isnull().sum().sort_values(ascending=False)

The numerical features missing values are BuildingArea, YearBuilt, and Car. Since the total number of rows is 13580, BuildingArea is missing about 47% of its values (6450/13580) and YearBuilt about 40% (5375/13580), making any baseline imputation strategy unreliable. Thus, we remove these features from the dataset.

In [ ]:
melb_housing = melb_housing.drop(columns=["BuildingArea", "YearBuilt"])
X_train = X_train.drop(columns=["BuildingArea", "YearBuilt"])
X_test  = X_test.drop(columns=["BuildingArea", "YearBuilt"])

In [ ]:
plt.hist(melb_housing["Car"], color = '#8c0e50')
plt.title("Car Histogram")
plt.xlabel("Number of Carspots")
plt.ylabel("Frequency")
plt.show()

This is a histogram showing the distribution of carspots which is one of the numerical features with missing values (that we saw above). It is slightly right-skewed and has outliers; we have decided to impute missing values using the median, which is robust to outliers and avoids assumptions about the distribution.

In [ ]:
# Categorical Features 
print(melb_housing["CouncilArea"].shape[0])

#check which categorical columns are missing values and how many
categorical_cols = melb_housing.select_dtypes(include=['object']).columns
melb_housing[categorical_cols].isnull().sum().sort_values(ascending=False)

Here we see that the only categorical feature missing values is CouncilArea with 1369. We will impute missing values using the most frequent value to ensure that no category remains missing before encoding.

In [ ]:
# Imputation of missing values and encoding of categorical variables
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# remaining numerical columns
numerical_cols = melb_housing.select_dtypes(include=['int64', 'float64']).columns
numerical_cols = numerical_cols.drop('Price')

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categ_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categ_transformer, categorical_cols)
    ]
)


### Perform Linear Regression

Perform Linear Regression on training data. Predict the output for the test dataset using the fitted model. Print the root mean squared error (RMSE) and R^2 value from Linear Regression for test data set.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

model_lin_reg = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("regressor", LinearRegression())
])

model_lin_reg.fit(X_train, y_train)
y_pred = model_lin_reg.predict(X_test)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE: ", rmse)

# R^2 (Coefficient of Determination)
r2 = r2_score(y_test, y_pred)
print("R^2: ", r2)

**R^2 on Model Performance:**
About 70% of the total variation in Price can be explained by the model. Since R^2 is close to 1, we can say that the model explains most variance.

**RMSE on Model Performance:**
On average, the model's predictions are off by about $345320, which is not very good. The model is not very precise.

### Perform K-Nearest Neighbors Regression

Perform K-Nearest Neighbors Algorithm for Regression using cross validation to tune hyperparameters.

Then, using the optimal hyperparameters, train a K-Nearest Neighbors Regressor on the entirety of the training data and predict output for the test dataset using the fitted model.

Print root mean squared error (RMSE) and R^2 from K-Nearest Neighbors Regression on the test datset.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

#kfolds previously initialized
k_values = list(range(1, 15))
rmse_scores = {}

for k in k_values:
    fold_rmse = []
    knn_model = Pipeline([
        ("preprocess", preprocessor),
        ("knn", KNeighborsRegressor(n_neighbors=k, n_jobs = -1))
    ])

    for train_idx, val_idx in kfolds.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        knn_model.fit(X_tr, y_tr)
        predictions = knn_model.predict(X_val)
        fold_rmse.append(np.sqrt(mean_squared_error(y_val, predictions)))

    rmse_scores[k] = np.mean(fold_rmse)

best_k = min(rmse_scores, key=rmse_scores.get)
print("best k: ", best_k)


knn_model_optimal = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("knn", KNeighborsRegressor(n_neighbors=best_k, n_jobs=-1))
])

knn_model_optimal.fit(X_train, y_train)
y_pred = knn_model_optimal.predict(X_test)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE: ", rmse)

# R^2 (Coefficient of Determination)
r2 = r2_score(y_test, y_pred)
print("R^2: ", r2)

**K-NN vs. Linear Regression:**

- Linear regression assumes a linear relationship between the features and the target variable. KNN regression does not assume a linear relationship and is able to handle non-linear relationships.
- KNN regression is more computationally expensive.
- KNN only takes one hyperparameter, k (it does not take any parameters like linear regression).

**Additional steps for KNN Regression:**

- KNN regression requires scaling of features before training. This is because KNN regression needs to calculate and compare distances between observations.
- For KNN regression we perform cross validation to tune the hyperparameter, k. We then train the model using the optimal k.

### Perform Decision Tree Regression 

Perform Decision Tree Regression using cross validation to tune any hyperparameters.

Then, using the optimal hyperparameters, train a Decision Tree Regressor on the entirety of the training data and predict output for the test dataset using the fitted model.

Print root mean squared error (RMSE) and R^2 from Decision Tree Regression on the test dataset.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

#kfolds previously initialized
d_values = list(range(1, 15))
rmse_scores = {}

for d in d_values:
    fold_rmse = []
    tree_model = Pipeline([
        ("preprocess", preprocessor),
        ("tree", DecisionTreeRegressor(max_depth=d, random_state= 1))
    ])

    for train_idx, val_idx in kfolds.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        tree_model.fit(X_tr, y_tr)
        predictions = tree_model.predict(X_val)
        fold_rmse.append(np.sqrt(mean_squared_error(y_val, predictions)))

    rmse_scores[d] = np.mean(fold_rmse)

best_d = min(rmse_scores, key=rmse_scores.get)
print("best max_depth: ", best_d)


tree_model_optimal = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("tree", DecisionTreeRegressor(max_depth=best_d, random_state=1))
])

tree_model_optimal.fit(X_train, y_train)
y_pred = tree_model_optimal.predict(X_test)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE: ", rmse)

# R^2 (Coefficient of Determination)
r2 = r2_score(y_test, y_pred)
print("R^2: ", r2)

**Decision Tree over Linear Regression and KNN**
_Advantages_: 
- Decision trees are easy to understand. We can trace how specific predictions were made.
- Decision trees work for non-linear relationships (unlike linear regression).
- Decision trees do not need feature scaling (unlike KNN).

_Disadvantages_:
- Decision trees are more prone to overfitting.
- Changes to the dataset, even if small, can produce very different trees since decision trees work by making the best choice at each node (greedy algorithm).
- Decision trees do not tend to generalize well.

### Perform Random Forest Regression
Perform Random Forest Regression using cross validation to tune any hyperparameters.

Then, using the optimal hyperparameters, train a Random Forest Regressor on the entirety of the training data and predict output for the test dataset using the fitted model.

Print root mean squared error (RMSE) and R^2 from Random Forest Regression on the test dataset.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

#kfolds previously initialized
n_values = [70, 100, 130, 160, 190, 220]
rmse_scores = {}

for n in n_values:
    fold_rmse = []
    forest_model = Pipeline([
        ("preprocess", preprocessor),
        ("rf", RandomForestRegressor(n_estimators=n, random_state= 1, n_jobs=-1))
    ])

    for train_idx, val_idx in kfolds.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        forest_model.fit(X_tr, y_tr)
        predictions = forest_model.predict(X_val)
        fold_rmse.append(np.sqrt(mean_squared_error(y_val, predictions)))

    rmse_scores[n] = np.mean(fold_rmse)

best_n = min(rmse_scores, key=rmse_scores.get)
print("best number of trees: ", best_n)


forest_model_optimal = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("rf", RandomForestRegressor(n_estimators=best_n, random_state= 1, n_jobs = -1))
])

forest_model_optimal.fit(X_train, y_train)
y_pred = forest_model_optimal.predict(X_test)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE: ", rmse)

# R^2 (Coefficient of Determination)
r2 = r2_score(y_test, y_pred)
print("R^2: ", r2)

**Why does a Random Forest ensure better results than a single Decision Tree?**
- Decision trees are powerful tools but are prone to overfitting. Random forests address this problem by combining the predictions of different trees. Each decision tree is trained on a random subset of the data and features, ensuring low correlation between the trees. Thus, random forests make more stable and accurate predictions and generalize better.

## 3. Feature Extraction and Selection
### Feature Extraction

Apply **feature extraction** in this section in order to derive at least 2 new features.

### Location Clusters
We use Lattitude and Longtitude to cluster the data. These clusters then become a new feature (LocationCluster). We believe that adding LocationCluster to our set of features will improve model performance since houses nearby tend to share similar value (Price).

*New feature:* 
- LocationCluster -> Categorical feature (integer)

In [ ]:
from sklearn.cluster import KMeans

coords_train = X_train[['Longtitude', 'Lattitude']]
kmeans = KMeans(n_clusters=9, random_state= 1, n_init='auto')

# fit the model on the training data only to avoid data snooping
kmeans.fit(coords_train)
X_train["LocationCluster"] = kmeans.predict(X_train[['Longtitude', 'Lattitude']])
X_test["LocationCluster"] = kmeans.predict(X_test[['Longtitude', 'Lattitude']])

### New Features from Date
We extract two new features from Date (SaleYear and SaleMonth). SaleYear will help us identify price trends over the years, while SaleMonth will allow us to identify seasonal differences.

*New features:* 
- SaleYear -> Numerical feature
- SaleMonth -> Categorical feature (1 - 12)

In [ ]:
# Add Date to the dataset again
X_train['Date'] = old_melb_housing.loc[X_train.index, 'Date']
X_test['Date']  = old_melb_housing.loc[X_test.index, 'Date']

X_train['Date'] = pd.to_datetime(X_train['Date'], format="%d/%m/%Y")
X_train['SaleYear'] = X_train['Date'].dt.year
X_train['SaleMonth'] = X_train['Date'].dt.month

X_test['Date'] = pd.to_datetime(X_test['Date'], format="%d/%m/%Y")
X_test['SaleYear'] = X_test['Date'].dt.year
X_test['SaleMonth'] = X_test['Date'].dt.month

- **K-means:** We used the Lattitude and Longtitude features to create clusters using the K-means algorithm. Each instance was assigned a LocationCluster category. Location is a major determinant of house price, by assigning each instance to a cluster, we generate a new feature that can help us predict house prices more effectively.
- **Derived from Date:** We extracted two new features from Date, the year of sale and the month of sale. There may exist seasonal patterns that could help predict house prices more effectively.

### Feature Selection

Using the original and extracted features as input, perform **feature selection** in this section (statistical, model-based, or iterative feature selection).

We perform statistical feature selection to remove clearly irrelevant features. We calculate the Pearson correlation coefficient (r) of the numerical variables and the target variable and remove features with low correlation to the target variable. We expect this to reduce noise and improve performance for models that use the distance metric.

In [ ]:
num_features = X_train.select_dtypes(include=['int64', 'float64']).columns
corrs = X_train[num_features].corrwith(y_train)
# very conservative -> safe to drop
low_corr_features = corrs[corrs.abs() < 0.05].index.tolist()
X_train_new = X_train.drop(columns=low_corr_features)
X_test_new = X_test.drop(columns=low_corr_features)

We selected statistical feature selection to remove features that are almost completely uncorrelated to the target variable. We seek to eliminate noise to make our models more readable and improve performance for models that use the distance metric such as KNN.

## 4. Preprocessing and Feature Engineered Models

### Preprocessing for the Feature Engineered Models

Now that we've removed/added features during feature engineering, we need to update the preprocessing.

In [ ]:
# numerical columns
numerical_cols = X_train_new.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train_new.select_dtypes(include=['object']).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categ_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categ_transformer, categorical_cols)
    ]
)

### Perform Linear Regression

Perform Linear Regression on **engineered training data**. Predict the output for the test dataset using the fitted model. Print the root mean squared error (RMSE) and R^2 value from Linear Regression for test data set.

In [ ]:
model_lin_reg = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("regressor", LinearRegression())
])

model_lin_reg.fit(X_train_new, y_train)
y_pred = model_lin_reg.predict(X_test_new)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE: ", rmse)

# R^2 (Coefficient of Determination)
r2 = r2_score(y_test, y_pred)
print("R^2: ", r2)

### Perform K-Nearest Neighbors Regression

Perform K-Nearest Neighbors Algorithm for Regression using cross validation to tune any hyperparameters.

Then, using the optimal hyperparameters, train a K-Nearest Neighbors Regressor on the entirety of the training data and predict output for the test dataset using the fitted model.

Print root mean squared error (RMSE) and R^2 from K-Nearest Neighbors Regression on the test datset.

In [ ]:
#kfolds previously initialized
k_values = list(range(1, 15))
rmse_scores = {}

for k in k_values:
    fold_rmse = []
    knn_model = Pipeline([
        ("preprocess", preprocessor),
        ("knn", KNeighborsRegressor(n_neighbors=k, n_jobs = -1))
    ])

    for train_idx, val_idx in kfolds.split(X_train):
        X_tr, X_val = X_train_new.iloc[train_idx], X_train_new.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        knn_model.fit(X_tr, y_tr)
        predictions = knn_model.predict(X_val)
        fold_rmse.append(np.sqrt(mean_squared_error(y_val, predictions)))

    rmse_scores[k] = np.mean(fold_rmse)

best_k = min(rmse_scores, key=rmse_scores.get)
print("best k: ", best_k)


knn_model_optimal = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("knn", KNeighborsRegressor(n_neighbors=best_k, n_jobs=-1))
])

knn_model_optimal.fit(X_train_new, y_train)
y_pred = knn_model_optimal.predict(X_test_new)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE: ", rmse)

# R^2 (Coefficient of Determination)
r2 = r2_score(y_test, y_pred)
print("R^2: ", r2)

### Perform Decision Tree Regression 

Perform Decision Tree Regression using cross validation to tune any hyperparameters.

Then, using the optimal hyperparameters, train a Decision Tree Regressor on the entirety of the training data and predict output for the test dataset using the fitted model.

Print root mean squared error (RMSE) and R^2 from Decision Tree Regression on the test dataset.

In [ ]:
#kfolds previously initialized
d_values = list(range(1, 15))
rmse_scores = {}

for d in d_values:
    fold_rmse = []
    tree_model = Pipeline([
        ("preprocess", preprocessor),
        ("tree", DecisionTreeRegressor(max_depth=d, random_state= 1))
    ])

    for train_idx, val_idx in kfolds.split(X_train):
        X_tr, X_val = X_train_new.iloc[train_idx], X_train_new.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        tree_model.fit(X_tr, y_tr)
        predictions = tree_model.predict(X_val)
        fold_rmse.append(np.sqrt(mean_squared_error(y_val, predictions)))

    rmse_scores[d] = np.mean(fold_rmse)

best_d = min(rmse_scores, key=rmse_scores.get)
print("best max_depth: ", best_d)


tree_model_optimal = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("tree", DecisionTreeRegressor(max_depth=best_d, random_state=1))
])

tree_model_optimal.fit(X_train_new, y_train)
y_pred = tree_model_optimal.predict(X_test_new)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE: ", rmse)

# R^2 (Coefficient of Determination)
r2 = r2_score(y_test, y_pred)
print("R^2: ", r2)

### Perform Random Forest Regression

Perform Random Forest Regression using cross validation to tune any hyperparameters.

Then, using the optimal hyperparameters, train a Random Forest Regressor on the entirety of the training data and predict output for the test dataset using the fitted model.

Print root mean squared error (RMSE) and R^2 from Random Forest Regression on the test dataset.

In [ ]:
#kfolds previously initialized
n_values = [70, 100, 130, 160, 190, 220]
rmse_scores = {}

for n in n_values:
    fold_rmse = []
    forest_model = Pipeline([
        ("preprocess", preprocessor),
        ("rf", RandomForestRegressor(n_estimators=n, random_state= 1, n_jobs=-1))
    ])

    for train_idx, val_idx in kfolds.split(X_train):
        X_tr, X_val = X_train_new.iloc[train_idx], X_train_new.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        forest_model.fit(X_tr, y_tr)
        predictions = forest_model.predict(X_val)
        fold_rmse.append(np.sqrt(mean_squared_error(y_val, predictions)))

    rmse_scores[n] = np.mean(fold_rmse)

best_n = min(rmse_scores, key=rmse_scores.get)
print("best number of trees: ", best_n)


forest_model_optimal = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("rf", RandomForestRegressor(n_estimators=best_n, random_state= 1, n_jobs = -1))
])

forest_model_optimal.fit(X_train_new, y_train)
y_pred = forest_model_optimal.predict(X_test_new)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE: ", rmse)

# R^2 (Coefficient of Determination)
r2 = r2_score(y_test, y_pred)
print("R^2: ", r2)

## 5. Model Comparision
Create some visualizations that compare all the models results using either $R^{2}$ or RMSE. Be sure to include the following visualizations:
1. Performance comparison across the 4 baseline models from problem 2.
2. Performance comparison across the 4 feature engineered models from problem 4.
3. Performance comparison of the baseline models and the feature engineered models.

Feel free to include additional visualizations!

# 1. Baseline Models Performance Comparison (Problem 2)

In [ ]:
sns.set_style('whitegrid')
plt.figure(figsize=(10,6))

#define input data for plotting. 
data = {'Model': ['Linear Regression', 'K-Nearest Neighbors Regressor','Decision Tree Regressor', 'Random Forest Regressor'], 
        'R^2': [0.6999702687645968, 0.7496379467598079, 0.7208037306305017,  0.8361249660679958]}

df_results = pd.DataFrame(data)

#sort DataFrame in descending order. 
df_r2_sorted = df_results.sort_values(by = 'R^2', ascending = False)

ax_r2 = sns.barplot(x = 'Model', y = 'R^2', data = df_results, palette = 'magma')

#plot characteristics
plt.title('Baseline Models Performance Comparison (Using $R^2$)', fontsize = 14)
plt.xlabel('Baseline Models', fontsize = 14)
plt.ylabel('$R^2$ (Coefficient of Determination)')
plt.xticks(rotation = 17)

df_results




# 2. Feature Engineered Models Performance Comparison (Problem 4)

In [ ]:
#define input data for plotting. 
data = {'Model': ['Linear Regression', 'K-Nearest Neighbors Regressor','Decision Tree Regressor', 'Random Forest Regressor'], 
        'R^2': [0.7236800514002257, 0.7490915649727414, 0.7127435005959251,  0.8355368267681276]}

df_results = pd.DataFrame(data)

sns.set_style('whitegrid')

plt.figure(figsize=(10,6))
df_r2_sorted = df_results.sort_values(by = 'R^2', ascending = False)

ax_r2 = sns.barplot(x = 'Model', y = 'R^2', data = df_results, palette = 'magma')

#plot characteristics. 
plt.title('Feature Engineered Models Performance Comparison (Using $R^2$)', fontsize = 14)
plt.xlabel('Feature Engineered Models', fontsize = 14)
plt.ylabel('$R^2$ (Coefficient of Determination)')
plt.xticks(rotation = 17)

df_results


# 3. Baseline Models vs. Feature Engineered Models

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(14, 8))


baseline_data = {'Model': ['Linear Regression', 'K-Nearest Neighbors Regressor','Decision Tree Regressor', 'Random Forest Regressor'], 
        '$R^2$': [0.6999702687645968, 0.7496379467598079, 0.7208037306305017,  0.8361249660679958], 'Model Type': ['Baseline'] * 4}

feat_eng_data = {'Model': ['Linear Regression', 'K-Nearest Neighbors Regressor','Decision Tree Regressor', 'Random Forest Regressor'], 
        '$R^2$': [0.7236800514002257, 0.7490915649727414, 0.7127435005959251,  0.8355368267681276], 'Model Type': ['Feature Engineered'] * 4}

#creating and combining to one plot together.
df_baseline = pd.DataFrame(baseline_data)
df_feat_eng_data = pd.DataFrame(feat_eng_data)
df_combined = pd.concat([df_baseline, df_feat_eng_data])

df_combined_sorted = df_combined.sort_values(by=['Model', '$R^2$'])


model_order = ['Linear Regression','K-Nearest Neighbors Regressor', 'Decision Tree Regressor','Random Forest Regressor']

ax_r2 = sns.barplot(x = 'Model', y = '$R^2$', hue='Model Type', data = df_combined_sorted, palette = 'magma', order = model_order)


#customizations.
plt.title('Performance Comparison: Baseline vs. Feature Engineered Models (Using $R^2$)', fontsize=16)
plt.ylabel('$R^2$ (Coefficient of Determination)', fontsize=14)
plt.xlabel('Regression Algorithms', fontsize=14)
plt.legend(title='Model Type', loc='upper left')
plt.xticks(rotation=15, ha='right')


plt.tight_layout()
plt.show()

df_combined_sorted



# Summary
### 1. Baseline Models Performance Comparison (Problem 2) Summary
**Best Performing**: Looking at our Seaborn bar plots for our models without feature engineering, we can see that Random Forest Regressor is the best performing model with an R^2 score of about 0.835537 which conveys strong predictive performance. RF Regressor is followed behind by K-Nearest Neighbors Regressor (0.749638) and Decision Tree Regressor (0.720804). There seems to exist a perforance gap between the best peforming baseline model (RF) and the second and third-best performing baseline models (KNN and DT respectively).

**Interpretation**: The success RF Regressor saw was due to its ability and method to capture non-linear relationships between features and our target variable which is crucial in determining complex scenarios like house prices. Because it also uses bagging (bootstrap aggregation), it's able to minimize overfitting and reducing variance to produce a more stable, robust prediction. Additionally, RF's ability to handle feature importance implicitly (like Gini importance) means it can automatically identify which variables have the strongest influence on house prices, effectively performing feature selection during the training process.

**Worst Performing**: The worst performing model is Linear Regression with a R^2 score of 0.699970 -- the weakest out of all our four baseline models. This is due to the fact that it was highly susceptible to overfitting and high variance (when it's used alone on data like housing prices which can be noisy and complex). Not necessarily extremely poor, but falls shorter compared to the other models' respective performances. This performance gap reveals fundamental limitations in Linear Regression's ability to capture the complex patterns present in housing price data.

**Interpretation** The underperformance of Linear Regression can be caused by how the algorithm assumes an idea of linearity: a presumption that median house prices and our target variable can operate on a straight line on a constant slope. However, housing prices are complex scenarios and it seems like Linear Regression had a hard time picking up the more complex, non-linear patterns. Without explicit feature engineering, Linear Regression cannot capture these intricate relationships. The model's simplicity becomes a liability here by exhibiting high bias where it underfits data with restrictive functional forms.

### 2. Feature Engineered Models Performance Comparison (Problem 4) Summary
**Best Performing**: Random Forest Regressor seems to have performed the best when feature engineering was included with a R^2 score of about 0.835537. However, it did perform slightly worse compared to its baseline model. The algorithm is strong because it's able to capture non-linearity as well as complex feature interactions that is inherent to housing pricing. The bagging mechanism ensures that no single tree's prediction dominates, reducing variance and mitigating overfitting, which translates to robust generalization on unseen data. The slight drop in performance when including Feature Engineering could be perhaps that Feature Engineering provided limited incremental value; that could be a possibility for explaining the small drop in performance. Perhaps, the Feature Engineering did not add enough new information that the baseline model itself couldn't find and may have (possibly) added extra noise and complexity. 

**Worst Performing**: The worst performing was Decision Tree Regressor for feature engineering with a R^2 score of 0.712744. This is several points before the RF regressor which can suggest several limitations of the DT Regressor. With feature engineering, DT Regressor's vulnerabilities may have been exacerbated: additional engineered features increased the feature space dimensionality, giving the tree more opportunities to create overly specific splits that capture spurious patterns rather than the true, underlying relationships. The tree may have partitioned data based on complex engineered features that worked well on training data but fail to generalize on test data. 

### 3. Baseline Models vs. Feature Engineered Models Summary
**Benefited the Best**: Linear Regression seems to have benefitted the best out of all the models when comparing them side-by-side. By encoding complex patterns into the feature set, we compensated for the algorithm's rigid linearity assumption, allowing it to model relationships it couldn't discover independently which improved its model performance from its initial baseline performance, jumping from a R^2 score of 0.699970 to 0.723680 (when feature engineering).

**Benefitted the Least**: Decision Tree Regressor seemed to have benefitted the least from feature engineering as it performed worse than its baseline model: its score going from 0.720804 down to 0.712744. It seems to have been able to create more opportunites to overfit. Decision Trees can capture non-linearity (unlike Linear Regression) through things like their recursive partitioning; the engineered features seemed to have been redundant and might've added noise and complexity that worsened its performance.  

**Notable Insight:** All the models except Linear Regression seemed to have decreased in model performance compared to their baseline models. There could be several reasons for this. It is possible be that feature engineering did not add any new information that the models couldn't do themselves. The feature engineering, instead of improving the model performance, might've added noise and complexity.